# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.2 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = "task141"
CH = 10
H = W = 30
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
TASK_PATH = Path(COMPETITION)/f"{TASK_ID}.json"  # overridden below for local/task-json use
ONNX_PATH = Path(f"{TASK_ID}.onnx")
SUBMISSION_PATH = Path("submission.zip")

In [6]:
LOCAL_TASK_JSON = Path("/mnt/data/task141.json")
if LOCAL_TASK_JSON.exists():
    task = json.loads(LOCAL_TASK_JSON.read_text())
elif Path("task141.json").exists():
    task = json.loads(Path("task141.json").read_text())
else:
    task = None

def grid_to_tensor(grid):
    arr = np.asarray(grid, dtype=np.int64)
    x = np.zeros((1, CH, H, W), dtype=np.float32)
    h, w = arr.shape
    for c in range(CH):
        x[0, c, :h, :w] = (arr == c)
    return x

def padded_expected(grid):
    arr = np.asarray(grid, dtype=np.int64)
    y = np.zeros((H, W), dtype=np.int64)
    h, w = arr.shape
    y[:h, :w] = arr
    return y

def pred_grid(session, grid):
    y = session.run(None, {session.get_inputs()[0].name: grid_to_tensor(grid)})[0]
    return y.argmax(axis=1)[0].astype(np.int64)

In [7]:
class Task141CanvasMaskModel(nn.Module):
    def __init__(self):
        super().__init__()
        rr = torch.arange(H, dtype=torch.float32).view(1, H, 1).expand(1, H, W)
        cc = torch.arange(W, dtype=torch.float32).view(1, 1, W).expand(1, H, W)
        colors = torch.arange(CH, dtype=torch.float32).view(1, CH, 1, 1)
        self.register_buffer("rr", rr)
        self.register_buffer("cc", cc)
        self.register_buffer("colors", colors)

    def forward(self, x):
        # Real cells are one-hot; padded cells are all-zero. This recovers the true canvas.
        active = (torch.sum(x, dim=1) > 0.5).float()
        fg = torch.sum(x[:, 1:, :, :], dim=1)
        r0 = torch.sum(fg * self.rr).reshape(1, 1, 1)
        c0 = torch.sum(fg * self.cc).reshape(1, 1, 1)
        color_val = torch.sum(x * self.colors).reshape(1, 1, 1, 1)
        diag = (torch.abs(self.rr - r0) - torch.abs(self.cc - c0)).abs()
        draw = (diag < 0.25).float() * active

        outs = []
        for c in range(CH):
            ch = x[:, c, :, :]
            if c == 0:
                ch = ch * (1.0 - draw)
            else:
                color_flag = (torch.abs(color_val - float(c)) < 0.25).float().reshape(1, 1, 1)
                ch = torch.maximum(ch, draw * color_flag)
            outs.append(ch)
        return torch.stack(outs, dim=1)

model = Task141CanvasMaskModel().eval()

In [8]:
dummy = np.zeros((1, CH, H, W), dtype=np.float32)
dummy[0, 0, :17, :17] = 1.0
dummy[0, 0, 7, 12] = 0.0
dummy[0, 6, 7, 12] = 1.0

torch.onnx.export(
    model,
    torch.tensor(dummy),
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

m = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(m)
m = shape_inference.infer_shapes(m)
onnx.save(m, str(ONNX_PATH))
print("saved", ONNX_PATH, "bytes", ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/3643615142.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


saved task141.onnx bytes 26881


In [9]:
m = onnx.load(str(ONNX_PATH))
ops = sorted({node.op_type for node in m.graph.node})
print("ops:", ops)
print("forbidden:", sorted(set(ops) & FORBIDDEN_OPS))
print("empty optional inputs:", [(n.name, n.op_type) for n in m.graph.node for i in n.input if i == ""])

def dims(v):
    return [d.dim_value if d.HasField("dim_value") else None for d in v.type.tensor_type.shape.dim]
print("input shape:", dims(m.graph.input[0]))
print("output shape:", dims(m.graph.output[0]))
bad = []
for v in list(m.graph.input) + list(m.graph.output) + list(m.graph.value_info):
    ds = dims(v)
    if not ds or any(d in (None, 0) for d in ds):
        bad.append((v.name, ds))
print("bad static shapes:", bad[:5], "count", len(bad))

ops: ['Abs', 'Cast', 'Concat', 'Constant', 'Gather', 'Greater', 'Less', 'Max', 'Mul', 'ReduceSum', 'Reshape', 'Slice', 'Sub', 'Unsqueeze']
forbidden: []
empty optional inputs: []
input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
bad static shapes: [('/Constant_output_0', []), ('/Constant_1_output_0', []), ('/Constant_2_output_0', []), ('/Constant_3_output_0', []), ('/Constant_8_output_0', [])] count 34


In [10]:
if task is not None:
    sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
    for split in ["train", "test", "arc-gen"]:
        ok = total = 0
        for ex in task.get(split, []):
            pred = pred_grid(sess, ex["input"])
            exp = padded_expected(ex["output"])
            ok += bool(np.array_equal(pred, exp))
            total += 1
        print(f"{split}_raw_padded: {ok}/{total}")

In [11]:
with zipfile.ZipFile(SUBMISSION_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")
print("wrote", SUBMISSION_PATH)

wrote submission.zip
